# Multimodal Fusion System — End-to-End Inference Demo

This notebook walks through the full pipeline using **mock data** so it runs
without a real GraphCast JAX environment or a dataset.

## What this notebook demonstrates

1. Loading the trained satellite classifier (`cyclone.pth`) with `SatelliteWrapper`
2. Extracting the penultimate satellite embedding (512-d)
3. Running the `GraphCastWrapper` (mock mode) to get +6h/+12h/+18h/+24h forecasts
4. Cropping the local atmospheric window around the cyclone centre
5. Encoding atmospheric features with `AtmosphericEncoder`
6. Fusing all modalities with `MultimodalFusionModel`
7. Getting track, intensity, and confidence predictions
8. Viewing the structured JSON forecast output
9. Building dashboard-ready visualisation payloads

> **Note**: When you have a trained fusion checkpoint, pass its path to
> `CycloneForecaster(fusion_checkpoint=...)`. For real GraphCast, pass your
> JAX runner to `CycloneForecaster(graphcast_runner_fn=your_runner)`.

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Make sure the project root is on the path
PROJECT_ROOT = Path(".").resolve().parent   # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Files:", [f.name for f in PROJECT_ROOT.iterdir() if f.is_dir()])

In [ ]:
# ── 1. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
from PIL import Image
import json

from config.config_loader import load_config, get_device
from data.preprocessing import preprocess_image, get_inference_transform
from models.satellite_wrapper import SatelliteWrapper
from models.graphcast_wrapper import GraphCastWrapper
from models.atmospheric_encoder import AtmosphericEncoder
from models.fusion_model import MultimodalFusionModel
from models.track_head import TrackHead
from models.intensity_head import IntensityHead
from models.confidence_head import ConfidenceHead
from utils.feature_extraction import AtmosphericFeatureExtractor
from utils.geo import crop_atmospheric_window, encode_location
from utils.visualization import build_full_dashboard_payload

print("✓ All imports successful")

In [ ]:
# ── 2. Configuration ─────────────────────────────────────────────────────────
cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
device = get_device(cfg["device"])

print(f"Device: {device}")
print(f"Satellite classes: {cfg['satellite']['class_names']}")
print(f"Forecast horizons: {cfg['graphcast']['forecast_horizons_h']}h")
print(f"Crop window: ±{cfg['crop']['window_deg']}°")

## Step 1 — Satellite Classifier

We load the trained ResNet18 with `SatelliteWrapper`. A **forward hook** on the
`avgpool` layer captures the 512-d penultimate embedding. The model is fully frozen.

In [ ]:
# ── 3. Load satellite classifier ────────────────────────────────────────────
MODEL_PATH = PROJECT_ROOT / cfg["satellite_model_path"]
print(f"Loading model from: {MODEL_PATH}")

sat_wrapper = SatelliteWrapper(
    model_path=MODEL_PATH,
    num_classes=cfg["satellite"]["num_classes"],
    device=device,
)

# Verify all params frozen
trainable = sum(p.numel() for p in sat_wrapper._model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in sat_wrapper._model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} (should be 0)")

In [ ]:
# ── 4. Create a synthetic satellite image (replace with real INSAT image) ───
# In production: img = Image.open('path/to/cyclone.jpg').convert('RGB')
fake_pil = Image.fromarray(
    (np.random.rand(256, 256, 3) * 255).astype(np.uint8)
)

transform = get_inference_transform(cfg["satellite"]["input_size"])
img_tensor = transform(fake_pil).unsqueeze(0)  # (1, 3, 224, 224)

print(f"Image tensor shape: {img_tensor.shape}")
print(f"dtype: {img_tensor.dtype}")

In [ ]:
# ── 5. Run satellite classifier ──────────────────────────────────────────────
sat_out = sat_wrapper(img_tensor)

print("=== Satellite Classifier Output ===")
print(f"Predicted class:       {sat_out['predicted_class_name'][0]}")
print(f"Cyclone probability:   {sat_out['cyclone_probability'][0].item():.4f}")
print(f"Class probabilities:   {sat_out['probabilities'][0].tolist()}")
print(f"Satellite embedding:   shape={sat_out['satellite_embedding'].shape}")

satellite_embedding   = sat_out["satellite_embedding"]    # (1, 512)
cyclone_probability   = sat_out["cyclone_probability"]     # (1,)
predicted_class_name  = sat_out["predicted_class_name"][0]

## Step 2 — GraphCast Forecast (Mock Mode)

The `GraphCastWrapper` uses a **mock runner** here. To use your real GraphCast:

```python
def my_graphcast_runner(inputs):
    # Your JAX GraphCast call from graph.ipynb
    return preds  # xr.Dataset

gc_wrapper = GraphCastWrapper(runner_fn=my_graphcast_runner)
```

In [ ]:
# ── 6. GraphCast forecast ────────────────────────────────────────────────────
gc_wrapper = GraphCastWrapper(
    runner_fn=None,   # None → uses built-in mock runner
    variables=cfg["graphcast"]["variables"],
    forecast_horizons_h=cfg["graphcast"]["forecast_horizons_h"],
)

# In production, pass your real GDAS xr.Dataset here
forecasts = gc_wrapper.run_forecast(inputs=None)

print("GraphCast forecast horizons:", list(forecasts.keys()))
for h, ds in forecasts.items():
    if hasattr(ds, 'data_vars'):
        print(f"  {h}: variables={list(ds.data_vars)[:4]}...")

## Step 3 — Atmospheric Window Crop & Feature Extraction

In [ ]:
# ── 7. Crop atmospheric window ──────────────────────────────────────────────
CYCLONE_LAT = 20.4   # Replace with real cyclone centre
CYCLONE_LON = 88.1

window_deg  = cfg["crop"]["window_deg"]
grid_res    = cfg["crop"]["grid_resolution"]
n_spatial   = int(round(2 * window_deg / grid_res)) + 1

print(f"Cyclone centre: ({CYCLONE_LAT}°N, {CYCLONE_LON}°E)")
print(f"Crop window: ±{window_deg}° → {n_spatial}×{n_spatial} grid points")

crops = {}
for h in ["6h", "12h", "18h", "24h"]:
    crops[h] = crop_atmospheric_window(
        forecasts[h], CYCLONE_LAT, CYCLONE_LON, window_deg
    )

print("\nCropped datasets:")
for h, crop in crops.items():
    if hasattr(crop, 'sizes'):
        print(f"  {h}: sizes={dict(crop.sizes)}")

In [ ]:
# ── 8. Feature extraction ────────────────────────────────────────────────────
feature_extractor = AtmosphericFeatureExtractor(
    surface_vars=[
        "mean_sea_level_pressure",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
    ],
    pressure_vars=[
        "temperature",
        "u_component_of_wind",
        "v_component_of_wind",
        "geopotential",
        "specific_humidity",
    ],
    pressure_levels=cfg["graphcast"]["pressure_levels"],
    spatial_size=n_spatial,
)

atm_feature_tensors = {}
for h in ["6h", "12h", "18h", "24h"]:
    t = feature_extractor.extract_tensor(crops[h])  # flat 1-D tensor
    atm_feature_tensors[h] = t.unsqueeze(0).to(device)  # add batch dim

feat_dim = atm_feature_tensors["6h"].shape[-1]
print(f"Flat atmospheric feature dimension: {feat_dim:,}")
print(f"  = {feature_extractor._n_channels} channels × {n_spatial}×{n_spatial} grid")

## Step 4 — Atmospheric Encoder

In [ ]:
# ── 9. Build and run atmospheric encoder ────────────────────────────────────
ae_cfg = cfg["atmospheric_encoder"]

atm_encoder = AtmosphericEncoder(
    input_dim=feat_dim,
    hidden_dim=ae_cfg["hidden_dim"],
    output_dim=ae_cfg["output_dim"],
    dropout=ae_cfg["dropout"],
).to(device).eval()

atm_embeds = atm_encoder.encode_horizons([
    atm_feature_tensors["6h"],
    atm_feature_tensors["12h"],
    atm_feature_tensors["18h"],
    atm_feature_tensors["24h"],
])

print("Atmospheric embeddings per horizon:")
for h, emb in zip(["6h", "12h", "18h", "24h"], atm_embeds):
    print(f"  {h}: shape={emb.shape}")

## Step 5 — Multimodal Fusion

In [ ]:
# ── 10. Build fusion model and prediction heads ──────────────────────────────
f_cfg = cfg["fusion"]
h_cfg = cfg["heads"]

fusion_model = MultimodalFusionModel(
    satellite_embed_dim=cfg["satellite"]["embedding_dim"],
    atm_embed_dim=ae_cfg["output_dim"],
    sat_proj_dim=f_cfg["satellite_proj_dim"],
    atm_proj_dim=f_cfg["atm_proj_dim"],
    shared_hidden_dim=f_cfg["shared_hidden_dim"],
    use_location_embedding=f_cfg["use_location_embedding"],
    location_embed_dim=f_cfg["location_embed_dim"],
    dropout=0.0,  # disable dropout at inference
).to(device).eval()

track_head      = TrackHead(fusion_model.output_dim,      hidden_dim=h_cfg["track"]["hidden_dim"]).to(device).eval()
intensity_head  = IntensityHead(fusion_model.output_dim,  hidden_dim=h_cfg["intensity"]["hidden_dim"]).to(device).eval()
confidence_head = ConfidenceHead(fusion_model.output_dim, hidden_dim=h_cfg["confidence"]["hidden_dim"]).to(device).eval()

total_params = sum(
    sum(p.numel() for p in m.parameters())
    for m in [atm_encoder, fusion_model, track_head, intensity_head, confidence_head]
)
print(f"Total fusion system parameters: {total_params:,}")

In [ ]:
# ── 11. Run fusion forward pass ──────────────────────────────────────────────
loc_enc  = encode_location(CYCLONE_LAT, CYCLONE_LON).unsqueeze(0).to(device)
sat_prob = cyclone_probability.unsqueeze(-1).to(device)  # (1, 1)

with torch.no_grad():
    shared_repr = fusion_model(
        satellite_embedding=satellite_embedding.to(device),
        cyclone_probability=sat_prob,
        atm_embeddings=atm_embeds,
        location_encoding=loc_enc,
    )

    pred_track      = track_head(shared_repr)       # (1, 4, 2)
    pred_intensity  = intensity_head(shared_repr)   # (1, 4, 2)
    pred_confidence = confidence_head(shared_repr)  # (1, 1)

print(f"Shared representation: {shared_repr.shape}")
print(f"Track output:          {pred_track.shape}  (batch, horizons, lat/lon)")
print(f"Intensity output:      {pred_intensity.shape}  (batch, horizons, wind/pres)")
print(f"Confidence output:     {pred_confidence.shape}  (batch, 1)")

## Step 6 — Decode Predictions to Physical Units

In [ ]:
# ── 12. Decode outputs ───────────────────────────────────────────────────────
from models.intensity_head import WIND_MEAN, WIND_STD, PRES_MEAN, PRES_STD

HORIZONS = ["6h", "12h", "18h", "24h"]

track_deltas = pred_track[0].cpu()    # (4, 2)
intensity    = pred_intensity[0].cpu()  # (4, 2)
confidence   = pred_confidence[0, 0].item()

print("\n=== Decoded Forecast ===")
forecast_dict = {}

for i, h in enumerate(HORIZONS):
    pred_lat  = round(CYCLONE_LAT + track_deltas[i, 0].item(), 4)
    pred_lon  = round(CYCLONE_LON + track_deltas[i, 1].item(), 4)
    pred_wind = round(intensity[i, 0].item() * WIND_STD + WIND_MEAN, 1)
    pred_pres = round(intensity[i, 1].item() * PRES_STD + PRES_MEAN, 1)

    forecast_dict[h] = {
        "lat":      pred_lat,
        "lon":      pred_lon,
        "wind":     pred_wind,
        "pressure": pred_pres,
    }

    print(f"  +{h:>3s}: ({pred_lat:6.2f}°N, {pred_lon:6.2f}°E) | "
          f"Wind: {pred_wind:5.1f} kt | Pressure: {pred_pres:.1f} hPa")

print(f"\nForecast confidence: {confidence:.4f}")

In [ ]:
# ── 13. Assemble structured JSON result ────────────────────────────────────
result = {
    "cyclone_detected":    True,
    "cyclone_probability": round(cyclone_probability[0].item(), 4),
    "predicted_class":     predicted_class_name,
    "current_location": {
        "lat": CYCLONE_LAT,
        "lon": CYCLONE_LON,
    },
    "forecast":            forecast_dict,
    "forecast_confidence": round(confidence, 4),
    "error":               None,
}

print(json.dumps(result, indent=2))

## Step 7 — Dashboard Payload

The `build_full_dashboard_payload()` function packages everything into a format
ready for Streamlit, React, or any REST API.

In [ ]:
# ── 14. Build dashboard payload ──────────────────────────────────────────────
payload = build_full_dashboard_payload(result)

print("=== Dashboard Payload Keys ===")
for k, v in payload.items():
    if isinstance(v, dict):
        print(f"  {k}: {list(v.keys())}")
    elif isinstance(v, list):
        print(f"  {k}: list of {len(v)} items")
    else:
        print(f"  {k}: {v}")

print("\n--- GeoJSON track features ---")
for feat in payload["track_geojson"]["features"]:
    print(f"  {feat['geometry']['type']}: {feat.get('properties', {}).get('label', feat.get('properties', {}).get('name', ''))}")  

print("\n--- Intensity timeseries ---")
ts = payload["intensity_timeseries"]
for label, wind, pres in zip(ts["labels"], ts["wind_kt"], ts["pressure_hPa"]):
    print(f"  {label:>4s}: wind={wind}, pres={pres}")

print("\n--- Atmospheric cards ---")
for card in payload["atmospheric_cards"]:
    print(f"  {card['label']}: wind={card['wind_kt']} kt | {card['category']}")

## Step 8 — Using the One-Line API

Once you have a trained checkpoint, the entire pipeline collapses to:

In [ ]:
# ── 15. One-line inference API ───────────────────────────────────────────────
from inference.forecast import CycloneForecaster

forecaster = CycloneForecaster(
    config_path=PROJECT_ROOT / "config" / "config.yaml",
    fusion_checkpoint=None,          # Replace with path to best_fusion.pt
    graphcast_runner_fn=None,        # Replace with your JAX runner function
)

# Inference call
result = forecaster.forecast(
    satellite_image=fake_pil,        # PIL.Image or preprocessed tensor
    graphcast_input=None,            # xr.Dataset (None → mock runner)
    cyclone_lat=CYCLONE_LAT,
    cyclone_lon=CYCLONE_LON,
)

print("One-line API result:")
print(json.dumps(result, indent=2))

---

## Summary

| Component | Status | Notes |
|---|---|---|
| `SatelliteWrapper` | ✅ | Loads `cyclone.pth`, extracts 512-d embedding |
| `GraphCastWrapper` | ✅ | Mock mode; inject real JAX runner |
| `AtmosphericFeatureExtractor` | ✅ | Local crop → flat tensor |
| `AtmosphericEncoder` | ✅ | 3-layer MLP, shared across horizons |
| `MultimodalFusionModel` | ✅ | Concat+MLP, cross-attention ready |
| `TrackHead` | ✅ | (batch, 4, 2) — Δlat/Δlon |
| `IntensityHead` | ✅ | (batch, 4, 2) — wind/pressure |
| `ConfidenceHead` | ✅ | (batch, 1) — [0,1] |
| Visualization | ✅ | GeoJSON + timeseries + cards |

### Next steps

1. **Prepare training data** — fill `dataset/labels.csv` with aligned satellite+GraphCast samples
2. **Run training**: `python training/train_fusion.py --config config/config.yaml`
3. **Pass real GraphCast runner** to `CycloneForecaster(graphcast_runner_fn=...)`
4. **Plug into your dashboard** using `build_full_dashboard_payload(result)`